# Lab 06, part 2: the small files problem

The same 5,000 rows are written as 500 tiny files and as one file. Then we compact the tiny ones.

Why it matters: the NameNode keeps every file, directory and block in memory. Thousands of tiny files cost NameNode memory and make jobs start thousands of tasks for almost no data.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("lab-06-small-files")
    .config("spark.hadoop.dfs.replication", "1")
    .getOrCreate()
)
BASE = "hdfs://namenode:8020/user/lab/06/small"

def hdfs_summary(path):
    """Return (directories, files, bytes) under an HDFS path, like `hdfs dfs -count`."""
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    s = fs.getContentSummary(p)
    return s.getDirectoryCount(), s.getFileCount(), s.getLength()

In [ ]:
tiny = spark.range(5_000).select(
    F.col("id").alias("order_id"),
    F.round(F.rand(3) * 100, 2).alias("amount"),
)

tiny.repartition(500).write.mode("overwrite").parquet(f"{BASE}/many")
tiny.coalesce(1).write.mode("overwrite").parquet(f"{BASE}/one")
print("written")

## Count the files

Each Parquet file is also at least one HDFS block, so the NameNode tracks roughly files + blocks + directories objects.

In [ ]:
import pandas as pd

def describe(name):
    dirs, files, size = hdfs_summary(f"{BASE}/{name}")
    return {"folder": name, "directories": dirs, "files": files, "bytes": size}

pd.DataFrame([describe("many"), describe("one")])

Same row count, very different cost. Check the read time too:

In [ ]:
import time

for name in ["many", "one"]:
    t0 = time.perf_counter()
    n = spark.read.parquet(f"{BASE}/{name}").count()
    print(f"{name:5} rows={n} seconds={time.perf_counter() - t0:.2f}")

## Compact

Read the 500 files and write them back as one. `coalesce(1)` merges partitions without a full shuffle. In real life you would aim for files near the HDFS block size (128 MB), not one tiny file.

In [ ]:
spark.read.parquet(f"{BASE}/many").coalesce(1).write.mode("overwrite").parquet(f"{BASE}/compacted")

print(describe("compacted"))
print("rows after compaction:", spark.read.parquet(f"{BASE}/compacted").count())

In [ ]:
spark.stop()